Objectives:
1. **GPT backbone** (token + positional embeddings → blocks → logits)
2. **LayerNorm** (stabilize training; why not BatchNorm for causal models)
3. **FeedForward + GELU**
4. **Residual / shortcut connections**
5. **Transformer block** (Pre-LN + masked multi-head attention + FFN)
6. **Final GPT model**
7. **Greedy text generation** (untrained demo)


---

## 0) Setup

We use:
- **PyTorch** for modules and tensor ops
- **tiktoken** for GPT-2 BPE tokenization (consistent with GPT-style models)


In [ ]:
# Core imports
import math  # for sqrt scaling, etc.
import torch  # tensors + autograd
import torch.nn as nn  # neural network layers
import torch.nn.functional as F  # functional ops (softmax, etc.)

# Tokenizer (GPT-2 BPE)
import tiktoken  # installed

# Reproducibility
torch.manual_seed(0)  # fix RNG seed so results are repeatable

# Device selection (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"  # choose device
print("Using device:", device)

In [ ]:
# Helper: parameter counting + rough memory estimates

def count_parameters(model: nn.Module) -> int:
    # Count all learnable parameters in a model
    return sum(p.numel() for p in model.parameters())

def bytes_to_megabytes(n_bytes: int) -> float:
    # Convert bytes to MB
    return n_bytes / (1024 ** 2)

def estimate_param_memory_bytes(model: nn.Module, bytes_per_param: int = 4) -> int:
    # FP32 params ~4 bytes; FP16/BF16 params ~2 bytes
    return count_parameters(model) * bytes_per_param

---

## 1) Configuration: "toy GPT" (fast) vs GPT-2 small (reference)

We’ll **demo** with a small config so everything runs instantly.

We’ll keep:
- `vocab_size = 50257` (GPT-2 BPE vocab)


In [ ]:
# Reference tokenizer (GPT-2 BPE)
enc = tiktoken.get_encoding("gpt2")  # GPT-2 encoding
VOCAB_SIZE = enc.n_vocab  # 50257

# A tiny config
GPT_CONFIG_TOY = {
    "vocab_size": VOCAB_SIZE,   # BPE vocabulary size
    "context_length": 32,       # max tokens in a sequence
    "emb_dim": 64,              # embedding dimension (tiny)
    "n_heads": 4,               # attention heads (must divide emb_dim)
    "n_layers": 2,              # number of transformer blocks
    "drop_rate": 0.1,           # dropout probability
    "qkv_bias": False           # whether Q/K/V projections have bias
}

# A reference (not used for heavy computation in this lab)
GPT_CONFIG_GPT2_SMALL = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

print("Toy config:", GPT_CONFIG_TOY)

---

## 2) Step 1 — GPT backbone (DummyGPTModel)

**Goal:** Understand the **shape flow** end-to-end before adding real transformer logic.

Backbone components:
- token embeddings + positional embeddings
- dropout
- transformer blocks (placeholder for now)
- final normalization (placeholder for now)
- linear output head → **logits** over vocabulary

Input:
- token IDs: `[B, T]`

Output:
- logits: `[B, T, V]`


In [ ]:
# Step 1: placeholder modules so we can test the pipeline quickly

class DummyTransformerBlock(nn.Module):
    # A placeholder that does nothing (identity)
    def __init__(self, cfg):
        super().__init__()  # initialize nn.Module

    def forward(self, x):
        return x  # pass through unchanged

class DummyLayerNorm(nn.Module):
    # A placeholder that does nothing (identity)
    def __init__(self, cfg):
        super().__init__()  # initialize nn.Module

    def forward(self, x):
        return x  # pass through unchanged

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()  # initialize nn.Module

        # Token embeddings: [V, D]
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Positional embeddings: [context_length, D]
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # Dropout on embeddings
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Stack of placeholder transformer blocks
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Placeholder final norm
        self.final_norm = DummyLayerNorm(cfg)

        # Output head maps embedding dim -> vocabulary logits
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx):
        # idx: [B, T] token IDs
        B, T = idx.shape  # batch size and sequence length

        # Token embeddings: [B, T, D]
        tok = self.tok_emb(idx)

        # Positions: [T]
        pos_ids = torch.arange(T, device=idx.device)

        # Positional embeddings: [T, D] (broadcast to [B, T, D] when added)
        pos = self.pos_emb(pos_ids)

        # Combine embeddings: [B, T, D]
        x = tok + pos

        # Apply dropout
        x = self.drop_emb(x)

        # Pass through placeholder blocks and placeholder final norm
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        # Logits: [B, T, V]
        logits = self.out_head(x)

        return logits

In [ ]:
# Quick test: tokenize a prompt, run forward pass, check shapes

prompt = "Every effort moves you"  # classic demo prompt
ids = enc.encode(prompt)  # convert text -> BPE token IDs (list[int])

# Make a batch with B=1 and pad/truncate to T=context_length
T = GPT_CONFIG_TOY["context_length"]  # context length
ids = ids[:T]  # truncate if needed
if len(ids) < T:
    ids = ids + [enc.eot_token] * (T - len(ids))  # pad with end-of-text token

idx = torch.tensor([ids], dtype=torch.long, device=device)  # [B, T]

dummy = DummyGPTModel(GPT_CONFIG_TOY).to(device)  # create model
logits = dummy(idx)  # forward pass

print("idx shape:", idx.shape)        # [B, T]
print("logits shape:", logits.shape)  # [B, T, V]

---

## 3) Step 2 — Layer Normalization (LayerNorm) from scratch

**Why LayerNorm?**
Deep networks can suffer from unstable activations → unstable training.
LayerNorm normalizes each token’s feature vector (over the embedding dimension):
- mean → 0
- variance → 1

**Important:** We normalize across the **feature dimension** (`D`), not across the batch.

We'll implement a minimal LayerNorm:
- learnable scale `gamma`
- learnable shift `beta`


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim: int, eps: float = 1e-5):
        super().__init__()  # initialize base class

        self.eps = eps  # numerical stability term

        # Learnable parameters: scale (gamma) and shift (beta)
        self.gamma = nn.Parameter(torch.ones(emb_dim))  # [D]
        self.beta = nn.Parameter(torch.zeros(emb_dim))  # [D]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]

        # Compute mean over the feature dimension D
        mean = x.mean(dim=-1, keepdim=True)  # [B, T, 1]

        # Compute variance over the feature dimension D
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # [B, T, 1]

        # Normalize
        x_hat = (x - mean) / torch.sqrt(var + self.eps)  # [B, T, D]

        # Scale and shift (broadcast gamma/beta over B and T)
        out = self.gamma * x_hat + self.beta  # [B, T, D]

        return out

In [ ]:
# Sanity check: after LayerNorm, mean ~0 and var ~1 along the last dimension

x_demo = torch.randn(2, 4, GPT_CONFIG_TOY["emb_dim"], device=device)  # [B=2, T=4, D]

ln = LayerNorm(GPT_CONFIG_TOY["emb_dim"]).to(device)  # our LayerNorm
y_demo = ln(x_demo)  # normalized output

# Compute mean/var across D for a quick check
mean_after = y_demo.mean(dim=-1)  # [B, T]
var_after = y_demo.var(dim=-1, unbiased=False)  # [B, T]

print("Mean after (first row):", mean_after[0].data)
print("Var after  (first row):", var_after[0].data)

---

## 4) Step 3 — FeedForward module + GELU

In GPT-2 style blocks, the MLP expands the embedding dimension by ~4×:
- Linear: `D → 4D`
- GELU
- Linear: `4D → D`

GELU is smoother than ReLU and allows small negative contributions.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim: int, drop_rate: float):
        super().__init__()  # initialize

        # Two-layer MLP with GELU in between
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),  # expand features
            nn.GELU(),                        # nonlinearity
            nn.Linear(4 * emb_dim, emb_dim),  # project back to emb_dim
            nn.Dropout(drop_rate)             # dropout for regularization
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        return self.net(x)  # [B, T, D]

In [ ]:
# Quick test: FFN preserves shape [B, T, D]

ff = FeedForward(GPT_CONFIG_TOY["emb_dim"], GPT_CONFIG_TOY["drop_rate"]).to(device)
out_ff = ff(x_demo)

print("Input shape:", x_demo.shape)
print("Output shape:", out_ff.shape)

---

## 5) Step 4 — Residual (shortcut) connections

Residual connections add the input back to the output of a sublayer:
- `x = x + sublayer(x)`

Why it matters:
- provides alternate gradient paths
- helps stability in deep stacks of blocks


In [ ]:
# A tiny residual demo: compare with/without residual

proj = nn.Linear(GPT_CONFIG_TOY["emb_dim"], GPT_CONFIG_TOY["emb_dim"]).to(device)  # simple sublayer
x0 = torch.randn(2, 4, GPT_CONFIG_TOY["emb_dim"], device=device)  # input

y_no_res = proj(x0)          # without residual
y_res = x0 + proj(x0)        # with residual

print("x0 mean:", float(x0.mean()))
print("y_no_res mean:", float(y_no_res.mean()))
print("y_res mean:", float(y_res.mean()))

---

## 6) Step 5 — TransformerBlock (Pre-LN GPT-2 style)

A GPT-style transformer block typically uses **Pre-LayerNorm**:

1. `x = x + Attention(LN(x))`
2. `x = x + FFN(LN(x))`

We also need **causal masked multi-head attention** so tokens cannot attend to the future.
You implemented attention earlier; here we’ll use a compact version.


In [ ]:
class MultiHeadCausalSelfAttention(nn.Module):
    # Multi-head causal self-attention (scaled dot-product)
    def __init__(self, emb_dim: int, num_heads: int, context_length: int, drop_rate: float, qkv_bias: bool):
        super().__init__()  # init

        assert emb_dim % num_heads == 0  # ensure heads divide embedding dim

        self.emb_dim = emb_dim  # embedding dimension D
        self.num_heads = num_heads  # number of heads H
        self.head_dim = emb_dim // num_heads  # per-head dim

        # Linear projections for Q, K, V (each produces D features)
        self.Wq = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)
        self.Wk = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)
        self.Wv = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)

        # Output projection back to emb_dim
        self.out_proj = nn.Linear(emb_dim, emb_dim, bias=True)

        # Dropout on attention weights
        self.attn_drop = nn.Dropout(drop_rate)

        # Register a causal mask as a non-parameter buffer
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        B, T, D = x.shape  # unpack

        # Project to Q, K, V: [B, T, D]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Reshape into heads: [B, T, D] -> [B, H, T, head_dim]
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scores: [B, H, T, T]
        scores = Q @ K.transpose(-2, -1)

        # Scale scores for stability
        scores = scores / math.sqrt(self.head_dim)

        # Apply causal mask (slice to current T)
        mask = self.mask[:T, :T]
        scores = scores.masked_fill(mask, -torch.inf)

        # Softmax over last dim to get attention weights
        weights = torch.softmax(scores, dim=-1)

        # Apply dropout to attention weights
        weights = self.attn_drop(weights)

        # Context per head: [B, H, T, head_dim]
        context = weights @ V

        # Recombine heads: [B, H, T, head_dim] -> [B, T, D]
        context = context.transpose(1, 2).contiguous().view(B, T, D)

        # Final projection: [B, T, D]
        out = self.out_proj(context)

        return out

In [ ]:
class TransformerBlock(nn.Module):
    # GPT-2 style Pre-LN transformer block: (Attn + FFN) with residual connections
    def __init__(self, cfg):
        super().__init__()  # init

        D = cfg["emb_dim"]  # embedding dim

        # Pre-LN layers
        self.ln1 = LayerNorm(D)
        self.ln2 = LayerNorm(D)

        # Causal multi-head attention
        self.attn = MultiHeadCausalSelfAttention(
            emb_dim=D,
            num_heads=cfg["n_heads"],
            context_length=cfg["context_length"],
            drop_rate=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        # Feed-forward network
        self.ff = FeedForward(D, cfg["drop_rate"])

        # Dropout on residual branches (common in GPT-style)
        self.resid_drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]

        # Attention block (Pre-LN) + residual
        x = x + self.resid_drop(self.attn(self.ln1(x)))

        # Feed-forward block (Pre-LN) + residual
        x = x + self.resid_drop(self.ff(self.ln2(x)))

        return x

In [ ]:
# Quick sanity test: TransformerBlock preserves shape

B = 2  # batch size
T = GPT_CONFIG_TOY["context_length"]  # sequence length
D = GPT_CONFIG_TOY["emb_dim"]  # embedding dim

x_block = torch.randn(B, T, D, device=device)  # random input
block = TransformerBlock(GPT_CONFIG_TOY).to(device)  # block

y_block = block(x_block)  # forward

print("Input shape:", x_block.shape)
print("Output shape:", y_block.shape)

---

## 7) Step 6 — Full GPTModel (stack blocks → logits)

Now replace the dummy parts with real modules:
- token + positional embeddings
- dropout
- `n_layers` transformer blocks
- final LayerNorm
- linear output head → logits `[B, T, V]`

This is the model we use for next-token prediction.


In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()  # init

        self.cfg = cfg  # store cfg

        # Embeddings
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # Final normalization
        self.final_ln = LayerNorm(cfg["emb_dim"])

        # Output head (logits over vocab)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: [B, T]
        B, T = idx.shape  # unpack

        # Token embeddings: [B, T, D]
        tok = self.tok_emb(idx)

        # Positional embeddings: [T, D]
        pos_ids = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos_ids)

        # Combine + dropout: [B, T, D]
        x = self.drop_emb(tok + pos)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final norm
        x = self.final_ln(x)

        # Output logits: [B, T, V]
        logits = self.out_head(x)

        return logits

In [ ]:
# Create a GPT model and confirm shapes

model = GPTModel(GPT_CONFIG_TOY).to(device)  # create model

model.eval()  # disable dropout for deterministic demo

with torch.no_grad():  # no gradients needed for inference demo
    logits = model(idx)  # [B, T, V]

print("Logits shape:", logits.shape)  # [B, T, V]

# Parameter count + rough FP32 memory
n_params = count_parameters(model)
mem_fp32_mb = bytes_to_megabytes(estimate_param_memory_bytes(model, bytes_per_param=4))
print("Toy model parameters:", n_params)
print("Toy model param memory (FP32, MB):", round(mem_fp32_mb, 2))

---

## 8) Step 7 — Greedy text generation (untrained demo)

GPT generates text **one token at a time**:
1. Encode prompt → token IDs
2. Predict next-token logits
3. Choose next token (greedy = argmax)
4. Append token and repeat

Because our model is **untrained**, output will look random — that’s expected.


In [ ]:
def generate_text_simple(model: nn.Module, idx: torch.Tensor, max_new_tokens: int, context_size: int) -> torch.Tensor:
    # model: GPT-like model that returns logits [B, T, V]
    # idx: current token IDs [B, T_current]

    model.eval()  # ensure dropout is off

    for _ in range(max_new_tokens):  # generate tokens one-by-one

        # Crop context to the last context_size tokens
        idx_cond = idx[:, -context_size:]

        # Forward pass to get logits
        with torch.no_grad():
            logits = model(idx_cond)  # [B, T_cond, V]

        # Focus on last time step
        logits_last = logits[:, -1, :]  # [B, V]

        # Greedy choice: argmax over vocab
        next_id = torch.argmax(logits_last, dim=-1, keepdim=True)  # [B, 1]

        # Append next token to sequence
        idx = torch.cat([idx, next_id], dim=1)  # [B, T_current+1]

    return idx

In [ ]:
# Run generation starting from the prompt (untrained model)

model.eval()  # disable dropout

# Start from the *unpadded* prompt for generation
start_ids = enc.encode(prompt)  # list[int]
start = torch.tensor([start_ids], dtype=torch.long, device=device)  # [B=1, T0]

out_ids = generate_text_simple(
    model=model,
    idx=start,
    max_new_tokens=20,
    context_size=GPT_CONFIG_TOY["context_length"]
)

# Decode output to text
decoded = enc.decode(out_ids[0].tolist())

print("Prompt:", prompt)
print("\nGenerated text (untrained):")
print(decoded)

---

## Wrap-up

- A GPT model maps **token IDs → logits** over the vocabulary at each position: `[B, T] → [B, T, V]`
- The building blocks:
  - **LayerNorm** stabilizes activations (normalize over features)
  - **Causal multi-head attention** prevents seeing the future
  - **FeedForward (MLP)** expands `D → 4D → D` with **GELU**
  - **Residual connections** enable stable deep stacks
- **Text generation** repeats: encode → predict → select → append
